In [ ]:
import json
import numpy as np

with open("datasets/tiny_glove.json", "r", encoding="utf-8") as f:
    word_vectors = json.load(f)

print("词汇量:", len(word_vectors))
print("向量示例:")
print(word_vectors["good"])

词汇量: 4993
向量示例:
[-0.35585999488830566, 0.5213000178337097, -0.6107000112533569, -0.3013100028038025, 0.9486200213432312, -0.3153899908065796, -0.5983099937438965, 0.12188000231981277, -0.031943000853061676, 0.5569499731063843, -0.10621000081300735, 0.6339899897575378, -0.4733999967575073, -0.07589499652385712, 0.3824700117111206, 0.08156900107860565, 0.8221399784088135, 0.22220000624656677, -0.008376399986445904, -0.7662000060081482, -0.562529981136322, 0.6175900101661682, 0.20292000472545624, -0.048597998917102814, 0.8781499862670898, -1.6548999547958374, -0.774179995059967, 0.15434999763965607, 0.9482300281524658, -0.3952000141143799, 3.7302000522613525, 0.8285499811172485, -0.14103999733924866, 0.016395000740885735, 0.21115000545978546, -0.03608499839901924, -0.1558700054883957, 0.8658300042152405, 0.26309001445770264, -0.7101500034332275, -0.03677000105381012, 0.00182819995097816, -0.1770399957895279, 0.27031999826431274, 0.1102600023150444, 0.14133000373840332, -0.0573219992220401

In [ ]:
import json
import numpy as np

with open("/content/datasets/tiny_glove.json", "r", encoding="utf-8") as f:
    glove_vec = json.load(f)

print("词向量总单词数：", len(glove_vec))
print("good 词向量维度：", len(glove_vec["good"]))
print("good 向量前5维：", glove_vec["good"][:5])

词向量总单词数： 4993
good 词向量维度： 50
good 向量前5维： [-0.35585999488830566, 0.5213000178337097, -0.6107000112533569, -0.3013100028038025, 0.9486200213432312]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# 1. 构建词表和索引
vocab = list(glove_vec.keys())
word2idx = {word:i for i, word in enumerate(vocab)}
emb_dim = len(glove_vec[vocab[0]])

# 2. 构造预训练词向量矩阵
emb_matrix = torch.FloatTensor([glove_vec[w] for w in vocab])

# 3. 搭建模型：Embedding + 逻辑回归
class EmbeddingLR(nn.Module):
    def __init__(self, emb_matrix, emb_dim):
        super().__init__()
        # 加载预训练GloVe，不训练
        self.embedding = nn.Embedding.from_pretrained(emb_matrix, freeze=True)
        self.linear = nn.Linear(emb_dim, 1)

    def forward(self, x):
        # x: 单词索引序列
        embed = self.embedding(x)
        sent_emb = embed.mean(dim=1)  # 句子取平均
        out = self.linear(sent_emb)
        return torch.sigmoid(out)

# 初始化模型
model = EmbeddingLR(emb_matrix, emb_dim)
loss_fn = nn.BCELoss()
opt = optim.Adam(model.parameters(), lr=1e-3)

print("模型搭建完成")
print(model)

模型搭建完成
EmbeddingLR(
  (embedding): Embedding(4993, 50)
  (linear): Linear(in_features=50, out_features=1, bias=True)
)


In [ ]:
!pip install transformers

In [8]:
from transformers import pipeline
classifier = pipeline(
    "sentiment-analysis",
    model="siebert/sentiment-roberta-large-english",
    framework="pt"
)

# 测试两句
print("测试正面：", classifier("I love this movie so much!"))
print("测试负面：", classifier("This is the worst film ever."))

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


测试正面： [{'label': 'POSITIVE', 'score': 0.9988835453987122}]
测试负面： [{'label': 'NEGATIVE', 'score': 0.9995009899139404}]


In [10]:
# ==============================
# Task 3: 基于Embedding的逻辑回归分类器
# 完整训练版
# ==============================

import json
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 加载GloVe词向量
with open("/content/datasets/tiny_glove.json", "r", encoding="utf-8") as f:
    glove = json.load(f)

vocab = list(glove.keys())
word2idx = {word: i for i, word in enumerate(vocab)}
emb_dim = len(glove[vocab[0]])

# 2. 创建词向量矩阵
weights = torch.FloatTensor([glove[word] for word in vocab])

# 3. 构建模型
class EmbeddingLogisticRegression(nn.Module):
    def __init__(self, pretrained_weights, emb_dim):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(pretrained_weights, freeze=True)
        self.linear = nn.Linear(emb_dim, 1)

    def forward(self, x):
        emb = self.embedding(x)
        sent_vec = emb.mean(dim=1)
        out = self.linear(sent_vec)
        return torch.sigmoid(out)

# 4. 初始化模型
model = EmbeddingLogisticRegression(weights, emb_dim)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ----------------------
# 训练数据（简单示例）
# ----------------------
train_data = [
    ("i love this movie", 1),
    ("this is great good awesome", 1),
    ("i hate this film", 0),
    ("this is bad terrible worst", 0),
    ("amazing wonderful love", 1),
    ("awful boring hate", 0)
]

def sentence_to_idx(sentence):
    words = sentence.lower().split()
    idx = [word2idx[w] if w in word2idx else 0 for w in words]
    return torch.LongTensor([idx])

# ----------------------
# 训练开始
# ----------------------
print("开始训练...\n")
model.train()
for epoch in range(30):
    total_loss = 0
    for sent, label in train_data:
        x = sentence_to_idx(sent)
        y = torch.FloatTensor([[label]])

        pred = model(x)
        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d} | Loss: {total_loss:.4f}")

print("\n训练完成！")

# ----------------------
# 测试
# ----------------------
model.eval()
pos_sent = "i love this movie very good"
neg_sent = "i hate this film so bad"

pos_pred = model(sentence_to_idx(pos_sent))
neg_pred = model(sentence_to_idx(neg_sent))

print("\n===== 测试结果 =====")
print(f"正面句子预测概率: {pos_pred.item():.4f}")
print(f"负面句子预测概率: {neg_pred.item():.4f}")

print("\n✅ Task3 完全完成！")

开始训练...

Epoch  0 | Loss: 4.3666
Epoch  5 | Loss: 3.3700
Epoch 10 | Loss: 2.9744
Epoch 15 | Loss: 2.6507
Epoch 20 | Loss: 2.3735
Epoch 25 | Loss: 2.1340

训练完成！

===== 测试结果 =====
正面句子预测概率: 0.8862
负面句子预测概率: 0.4810

✅ Task3 完全完成！
